# RDFS, OWL 入門

- [RDF Schema 1.1](https://www.w3.org/TR/rdf-schema/)
- [OWL 2 Web Ontology Language Primer (Second Edition)](https://www.w3.org/TR/owl-primer/)
- [owlrl — OWL-RL 5.2.2 documentation](https://owl-rl.readthedocs.io/en/latest/owlrl.html)

## 準備

In [48]:
from rdflib import Dataset, Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph
from yfiles_jupyter_graphs import GraphWidget, Layout

from owlrl import (
    DeductiveClosure,
    RDFS_Semantics,
    OWLRL_Semantics,
    RDFS_OWLRL_Semantics,
)

EX = Namespace("https://example.org/ontology-demo/")


def show_rdf_graph(rdf_graph: Graph) -> GraphWidget:
    """Visualize an RDFLib graph with yFiles."""
    nx_graph = rdflib_to_networkx_multidigraph(
        rdf_graph,
        edge_attrs=lambda s, p, o: {
            "key": p,
            "label": p.n3(rdf_graph.namespace_manager),
        },
    )
    return GraphWidget(
        graph=nx_graph,
        graph_layout=Layout.HIERARCHICAL,
    )


## Schema と Data Graph

In [49]:
schema_ttl = """
@prefix ex: <https://example.org/ontology-demo/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Wallet rdfs:subClassOf ex:PortableObject .
ex:PortableObject rdfs:subClassOf ex:PhysicalObject .

ex:onTopOf rdfs:subPropertyOf ex:spatiallyRelatedTo ;
           rdfs:domain ex:MovableObject ;
           rdfs:range ex:SupportSurface .

ex:locatedIn rdfs:domain ex:PhysicalObject ;
             rdfs:range ex:Room .
"""

data_ttl = """
@prefix ex: <https://example.org/ontology-demo/> .

ex:wallet a ex:Wallet ;
          ex:onTopOf ex:table .

ex:table ex:locatedIn ex:kitchen .
"""

schema_graph = Graph().parse(data=schema_ttl, format="turtle")
facts_graph = Graph().parse(data=data_ttl, format="turtle")

# Schema と事実をひとつの明示グラフにまとめる
explicit_graph = Graph()
explicit_graph += schema_graph
explicit_graph += facts_graph

print("Schema triples:", len(schema_graph))
print("Fact triples:", len(facts_graph))
print("Explicit triples:", len(explicit_graph))


Schema triples: 7
Fact triples: 3
Explicit triples: 10


In [50]:
show_rdf_graph(schema_graph)


GraphWidget(layout=Layout(height='500px', width='100%'))

In [51]:
show_rdf_graph(facts_graph)


GraphWidget(layout=Layout(height='500px', width='100%'))

## 推論前に帰結を調べる

In [52]:
rdfs_targets = {
    "subClassOf": (EX.wallet, RDF.type, EX.PortableObject),
    "transitive subclass": (EX.wallet, RDF.type, EX.PhysicalObject),
    "subPropertyOf": (EX.wallet, EX.spatiallyRelatedTo, EX.table),
    "domain": (EX.wallet, RDF.type, EX.MovableObject),
    "range": (EX.table, RDF.type, EX.SupportSurface),
    "locatedIn range": (EX.kitchen, RDF.type, EX.Room),
}

for name, triple in rdfs_targets.items():
    print(name, triple in explicit_graph)

subClassOf False
transitive subclass False
subPropertyOf False
domain False
range False
locatedIn range False


## RDFS Closure

In [53]:
# 明示グラフをコピーしてから RDFS 推論を追加する
rdfs_closure = Graph()
rdfs_closure += explicit_graph

DeductiveClosure(
    RDFS_Semantics,  # RDFS に対応する推論規則を反復適用
    axiomatic_triples=False,
    datatype_axioms=False,
).expand(rdfs_closure)

for name, triple in rdfs_targets.items():
    print(name, triple in rdfs_closure)

assert all(t in rdfs_closure for t in rdfs_targets.values())


subClassOf True
transitive subclass True
subPropertyOf True
domain True
range True
locatedIn range True


In [54]:
show_rdf_graph(rdfs_closure)

GraphWidget(layout=Layout(height='690px', width='100%'))

## OWL の公理を追加

In [55]:
owl_ttl = """
@prefix ex: <https://example.org/ontology-demo/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:onTopOf owl:inverseOf ex:supports .

ex:assignedMarker a owl:FunctionalProperty .

ex:robot ex:assignedMarker ex:markerA, ex:markerB .

ex:markerA a ex:ReferenceMarker .
"""

owl_graph = Graph().parse(
    data=owl_ttl,
    format="turtle",
)

# 既存の明示グラフに OWL の公理を追加する
owl_explicit = Graph()
owl_explicit += explicit_graph
owl_explicit += owl_graph


In [56]:
show_rdf_graph(owl_graph)

GraphWidget(layout=Layout(height='500px', width='100%'))

In [57]:
show_rdf_graph(owl_explicit)

GraphWidget(layout=Layout(height='690px', width='100%'))

## RDFS と OWL-RL の比較

In [58]:
# 同じ OWL 入力グラフをそれぞれコピーし、推論方式ごとに展開する
rdfs_only = Graph()
rdfs_only += owl_explicit
DeductiveClosure(
    RDFS_Semantics,
    axiomatic_triples=False,
    datatype_axioms=False,
).expand(rdfs_only)

owlrl_only = Graph()
owlrl_only += owl_explicit
DeductiveClosure(
    OWLRL_Semantics,
    axiomatic_triples=False,
    datatype_axioms=False,
).expand(owlrl_only)

combined = Graph()
combined += owl_explicit
DeductiveClosure(
    RDFS_OWLRL_Semantics,
    axiomatic_triples=False,
    datatype_axioms=False,
).expand(combined)

owl_targets = {
    "inverseOf": (EX.table, EX.supports, EX.wallet),
    "FunctionalProperty -> sameAs": (EX.markerA, OWL.sameAs, EX.markerB),
    "sameAs substitution": (EX.markerB, RDF.type, EX.ReferenceMarker),
}

for name, triple in owl_targets.items():
    print(f"{name}:")
    print(f"  RDFS: {triple in rdfs_only}")
    print(f"  OWL-RL: {triple in owlrl_only}")
    print(f"  Combined: {triple in combined}")

assert all(triple in combined for triple in owl_targets.values())


inverseOf:
  RDFS: False
  OWL-RL: True
  Combined: True
FunctionalProperty -> sameAs:
  RDFS: False
  OWL-RL: True
  Combined: True
sameAs substitution:
  RDFS: False
  OWL-RL: True
  Combined: True


## SHACL との違い

In [59]:
from pyshacl import validate

shape_ttl = """
@prefix ex: <https://example.org/ontology-demo/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:RobotShape a sh:NodeShape ;
    sh:targetNode ex:robot ;
    sh:property [
        sh:path ex:assignedMarker ;
        sh:maxCount 1
    ] .
"""

shapes_graph = Graph().parse(
    data=shape_ttl,
    format="turtle",
)

conforms, report_graph, report_text = validate(
    owl_explicit,
    shacl_graph=shapes_graph,
    inference="none",
)

print("SHACL:", conforms)

print("OWL-RL sameAs:", (EX.markerA, OWL.sameAs, EX.markerB) in combined)

SHACL: False
OWL-RL sameAs: True


## Explicit / Entailed を分離

In [60]:
explicit_set = set(owl_explicit)
closure_set = set(combined)

entailed_set = closure_set - explicit_set

print("Explicit facts preserved:", explicit_set <= closure_set)

print("Newly materialized triples:", len(entailed_set))

Explicit facts preserved: True
Newly materialized triples: 430


In [61]:
for name, triple in owl_targets.items():
    print(name, triple in entailed_set)


inverseOf True
FunctionalProperty -> sameAs True
sameAs substitution True
